<a href="https://colab.research.google.com/github/DMULE2213/keras_tuner/blob/main/keral_hyper_parameter_tunning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [25]:
import numpy as np
import pandas as pd


In [29]:
df= pd.read_csv('/diabetes.csv')

In [30]:
 df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [31]:
df.corr()['Outcome']

,Outcome
Pregnancies,0.221898
Glucose,0.466581
BloodPressure,0.065068
SkinThickness,0.074752
Insulin,0.130548
BMI,0.292695
DiabetesPedigreeFunction,0.173844
Age,0.238356
Outcome,1.000000


In [32]:
x = df.iloc[:,:-1].values
y = df.iloc[:,-1].values

In [33]:
from sklearn.preprocessing import StandardScaler

In [34]:
scaler = StandardScaler()
x = scaler.fit_transform(x)

In [35]:
x.shape

(768, 8)

In [36]:
from sklearn.model_selection import train_test_split
x_train, x_test,y_train, y_test = train_test_split(x,y,test_size=0.2,random_state=0)


In [139]:
import tensorflow
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense, Dropout

In [38]:
model = Sequential()

model.add(Dense(32,activation='relu', input_dim=8))
model.add(Dense(1,activation='sigmoid'))

model.compile(optimizer='Adam',loss='binary_crossentropy', metrics=['accuracy'])


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [39]:
model.fit(x_train, y_train, batch_size=32, epochs=100, validation_data=(x_test, y_test))

Epoch 1/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 44ms/step - accuracy: 0.4442 - loss: 0.7898 - val_accuracy: 0.5519 - val_loss: 0.7262
Epoch 2/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.6294 - loss: 0.6920 - val_accuracy: 0.6299 - val_loss: 0.6681
Epoch 3/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6701 - loss: 0.6346 - val_accuracy: 0.6753 - val_loss: 0.6242
Epoch 4/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6634 - loss: 0.6086 - val_accuracy: 0.6948 - val_loss: 0.5895
Epoch 5/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6962 - loss: 0.5845 - val_accuracy: 0.7208 - val_loss: 0.5613
Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6963 - loss: 0.5769 - val_accuracy: 0.7208 - val_loss: 0.5385
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7468 - loss: 0.5276 - val_accuracy: 0.7338 - val_loss: 0.5199
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7539 - loss: 0.5213 - val_accuracy: 0.7403 - 

In [56]:
import keras_tuner as kt

In [60]:
tuner.search(x_train,y_train, epochs=5,validation_data=(x_test, y_test))

In [81]:
def build_model(hp):

  model = Sequential()

  model.add(Dense(32,activation='relu', input_dim=8))

  model.add(Dense(1,activation='sigmoid'))

  optimizer = hp.Choice('optimizer',values=['adam','sgd','rmsprop','adadelta']
                        )
  model.compile(optimizer= optimizer,loss='binary_crossentropy',metrics=['accuracy'])

  return model

In [82]:
tuner = kt.RandomSearch(build_model,objective='val_accuracy',max_trials=5)

Reloading Tuner from ./untitled_project/tuner0.json


In [83]:
tuner.search(x_train,y_train,epochs=15,validation_data=(x_test,y_test))

In [84]:
tuner.get_best_hyperparameters()[0].values

{'optimizer': 'rmsprop'}

In [85]:
model = tuner.get_best_models(num_models=1)[0]

In [90]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 321 (1.25 KB)

 Trainable params: 321 (1.25 KB)

 Non-trainable params: 0 (0.00 B)

In [91]:
model.fit(x_train,y_train,batch_size=32,epochs=100,initial_epoch=6,validation_data=(x_test,y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.7715 - loss: 0.4826 - val_accuracy: 0.7792 - val_loss: 0.4491
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7444 - loss: 0.5069 - val_accuracy: 0.7922 - val_loss: 0.4458
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7868 - loss: 0.4569 - val_accuracy: 0.7922 - val_loss: 0.4440
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7839 - loss: 0.4577 - val_accuracy: 0.8052 - val_loss: 0.4412
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7806 - loss: 0.4776 - val_accuracy: 0.8052 - val_loss: 0.4386
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - accuracy: 0.7820 - loss: 0.4615 - val_accuracy: 0.8052 - val_loss: 0.4374
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.7782 - loss: 0.4699 - val_accuracy: 0.8052 - val_loss: 0.4359
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.7883 - loss: 0.4568 - val_accuracy: 

In [96]:
def model_build(hp):

  model = Sequential()
  units = hp.Int('units',8, 128, step=8)
  model.add(Dense(units, activation='relu', input_dim=8))
  model.add(Dense(units,activation='relu'))
  model.add(Dense(1,activation='sigmoid'))

  model.compile(optimizer='rmsprop',loss='binary_crossentropy', metrics=['accuracy'])

  return model


In [97]:
tuner = kt.RandomSearch(model_build,objective='val_accuracy',max_trials=5, directory='mkdir', project_name = 'sumit')

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [100]:
tuner.search(x_train,y_train,epochs=5,validation_data=(x_test,y_test))

In [101]:
tuner.get_best_hyperparameters()[0].values

{'units': 72}

In [102]:
model = tuner.get_best_models(num_models=1)[0]

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.11/dist-packages/keras/src/saving/saving_lib.py:757: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 8 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [104]:
model.fit(x_train,y_train,batch_size=32,epochs=100,initial_epoch=6,validation_data=(x_test, y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step - accuracy: 0.7817 - loss: 0.4512 - val_accuracy: 0.8182 - val_loss: 0.4187
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7941 - loss: 0.4434 - val_accuracy: 0.8117 - val_loss: 0.4143
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7807 - loss: 0.4540 - val_accuracy: 0.8377 - val_loss: 0.4256
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7867 - loss: 0.4325 - val_accuracy: 0.8312 - val_loss: 0.4279
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7758 - loss: 0.4460 - val_accuracy: 0.8312 - val_loss: 0.4282
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7943 - loss: 0.4374 - val_accuracy: 0.8247 - val_loss: 0.4262
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7807 - loss: 0.4364 - val_accuracy: 0.8312 - val_loss: 0.4344
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7963 - loss: 0.4251 - val_accuracy: 0.82

In [110]:
def model_build(hp):

  model = Sequential()
  model.add(Dense(32, activation = 'relu', input_dim=8))

  for i in range(hp.Int('num_layer', min_value=1, max_value=10)):
    model.add(Dense(72, activation = 'relu'))

  model.add(Dense(1, activation = 'sigmoid'))

  model.compile(optimizer = 'rmsprop', loss = 'binary_crossentropy', metrics = ['accuracy'])

  return model



In [111]:
tuner= kt.RandomSearch(model_build, objective = 'val_accuracy', max_trials = 5, directory= 'mydir', project_name = 'sumit1')

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [112]:
tuner.search(x_train, y_train, epochs = 5, validation_data = (x_test, y_test))

Trial 5 Complete [00h 00m 05s]
val_accuracy: 0.8051947951316833

Best val_accuracy So Far: 0.8246753215789795
Total elapsed time: 00h 00m 24s


In [113]:
tuner.get_best_hyperparameters()[0].values

{'num_layer': 2}

In [114]:
tuner.get_best_models(num_models=1)[0]

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.11/dist-packages/keras/src/saving/saving_lib.py:757: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


<Sequential name=sequential, built=True>

In [116]:
model.fit(x_train, y_train, epochs=100, initial_epoch=5, validation_data=(x_test, y_test))

Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.8938 - loss: 0.2491 - val_accuracy: 0.7662 - val_loss: 0.5684
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.8867 - loss: 0.2686 - val_accuracy: 0.8182 - val_loss: 0.5627
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9061 - loss: 0.2479 - val_accuracy: 0.7922 - val_loss: 0.5786
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9136 - loss: 0.2501 - val_accuracy: 0.7857 - val_loss: 0.5697
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8922 - loss: 0.2790 - val_accuracy: 0.7922 - val_loss: 0.5969
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9063 - loss: 0.2507 - val_accuracy: 0.7857 - val_loss: 0.5861
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9272 - loss: 0.2327 - val_accuracy: 0.8117 - val_loss: 0.5682
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9092 - loss: 0.2267 - val_accuracy: 0.79

In [140]:
def build_model(hp):

  model = Sequential()


  counter = 0

  for i in range(hp.Int('num_layers', min_value=1, max_value=10)):
    if counter == 0:
      model.add(Dense(hp.Int('unit'+str(i), min_value=8, max_value=128, step =8),
      activation = hp.Choice ('activation'+ str(i), values = ['relu', 'tanh','sigmoid']),
                      input_dim = 8))
      model.add(Dropout(hp.Choice('Dropuot'+ str(i), values = [0.1, 0.2, 0.3, 0.4, 0.5,0.6, 0.7])))
  else:
     model.add(Dense(hp.Int('unit'+str(i), min_value=8, max_value=128, step =8),
     activation = hp.Choice ('activation'+ str(i), values = ['relu', 'tanh','sigmoid']),
                      ))
     model.add(Dropout(hp.Choice('Dropuot'+ str(i), values = [0.1, 0.2, 0.3, 0.4, 0.5,0.6, 0.7])))

  counter+=1
  model.add(Dense(1,activation = 'sigmoid'))

  model.compile(optimizer=hp.Choice('optimizer', values=['rmsprop','adam','sgd','nadam']),
                loss = 'binary_crossentropy',
                metrics = ['accuracy'])
  return model


In [142]:
tuner = kt.RandomSearch(build_model,
                        objective = 'val_accuracy',
                        max_trials = 3,
                        directory = 'mydir',
                        project_name = 'sumit4'
                        )


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [143]:
# tuner.search(x_train, y_train, epochs = 5, validation_data(x_test, y_test))
tuner.search(x_train, y_train, epochs=5, validation_data=(x_test, y_test))

Trial 3 Complete [00h 00m 06s]
val_accuracy: 0.6948052048683167

Best val_accuracy So Far: 0.6948052048683167
Total elapsed time: 00h 00m 15s


In [145]:
tuner.get_best_hyperparameters()[0].values

{'num_layers': 1,
 'unit0': 120,
 'activation0': 'sigmoid',
 'Dropuot0': 0.5,
 'optimizer': 'nadam'}

In [146]:
model = tuner.get_best_models(num_models = 1)[0]

/usr/local/lib/python3.11/dist-packages/keras/src/saving/saving_lib.py:757: UserWarning: Skipping variable loading for optimizer 'nadam', because it has 2 variables whereas the saved optimizer has 15 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [147]:
model.fit(x_train, y_train, batch_size = 3, epochs=100, initial_epoch=5, validation_data=(x_test, y_test) )

Epoch 6/100
205/205 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.5621 - loss: 0.7463 - val_accuracy: 0.6948 - val_loss: 0.5675
Epoch 7/100
205/205 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.6106 - loss: 0.6644 - val_accuracy: 0.7013 - val_loss: 0.5244
Epoch 8/100
205/205 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.6366 - loss: 0.6302 - val_accuracy: 0.7727 - val_loss: 0.4846
Epoch 9/100
205/205 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.6924 - loss: 0.5951 - val_accuracy: 0.7792 - val_loss: 0.4644
Epoch 10/100
205/205 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7147 - loss: 0.5665 - val_accuracy: 0.7857 - val_loss: 0.4544
Epoch 11/100
205/205 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7348 - loss: 0.5353 - val_accuracy: 0.7987 - val_loss: 0.4519
Epoch 12/100
205/205 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7577 - loss: 0.5002 - val_accuracy: 0.7727 - val_loss: 0.4684
Epoch 13/100
205/205 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7377 - loss: 0.5468 - val_

In [151]:
! git init

hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>
Initialized empty Git repository in /content/.git/
